# QueryMind ML — Exploratory Data Analysis

This notebook explores the raw + feature layers created by `ml_pipeline/sql/` and profiles the six ML activities defined in `database/neon_demo_setup.sql`:

| # | Activity | Type | Source view |
|---|----------|------|-------------|
| 1 | Tool adoption forecast | time-series regression | `ml_features_tool_adoption` |
| 2 | User adoption risk | multiclass classification | `ml_features_user_adoption_risk` |
| 3 | User segmentation | clustering | `ml_features_user_segments` |
| 4 | Usage anomaly detection | anomaly detection | `ml_features_usage_anomalies` |
| 5 | Team productivity forecast | time-series regression | `ml_features_team_productivity` |
| 6 | Tool recommendations | recommendation | `ml_features_tool_recommendations` |

**Prerequisites:** run `python run_pipeline.py --step sql` first so the raw + feature layers exist, and set `DATABASE_URL` (repo-root `.env`) to your Neon connection string.

The companion `01_training_walkthrough.ipynb` trains every activity; the same code path powers regular execution via `run_pipeline.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ML_ROOT))

from querymind_ml import config, db

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams["figure.figsize"] = (11, 4)

# Uses DATABASE_URL (repo .env) unless QM_DSN is provided.
conn = db.connect(config.get_dsn()).__enter__()

def q(sql: str) -> pd.DataFrame:
    return db.read_frame(conn, sql)

print("connected:", config.get_dsn().split("@")[-1])

## 0. Dataset health

Row counts and time coverage of everything the pipeline touches.

In [ ]:
q("""
SELECT 'ml_raw_usage_events' AS object, COUNT(*) AS rows FROM ml_raw_usage_events
UNION ALL SELECT 'ml_raw_weekly_usage', COUNT(*) FROM ml_raw_weekly_usage
UNION ALL SELECT '  -> injected anomaly cells', COUNT(*) FROM ml_raw_weekly_usage WHERE is_injected_anomaly
UNION ALL SELECT 'ml_raw_user_monthly_activity', COUNT(*) FROM ml_raw_user_monthly_activity
UNION ALL SELECT 'ml_raw_user_adoption_labels', COUNT(*) FROM ml_raw_user_adoption_labels
UNION ALL SELECT 'ml_raw_tool_adoption_actuals', COUNT(*) FROM ml_raw_tool_adoption_actuals
UNION ALL SELECT 'team_work_metrics', COUNT(*) FROM team_work_metrics
UNION ALL SELECT 'ai_tool_usage (app demo)', COUNT(*) FROM ai_tool_usage
ORDER BY rows DESC
""")

In [ ]:
q("""
SELECT batch_tag, COUNT(*) AS events,
       MIN(event_timestamp)::date AS first_event,
       MAX(event_timestamp)::date AS last_event,
       COUNT(DISTINCT user_id) AS users
FROM ml_raw_usage_events
GROUP BY batch_tag
ORDER BY events DESC
""")

In [ ]:
monthly = q("""
SELECT date_trunc('month', event_timestamp)::date AS month,
       COUNT(*) AS events,
       COUNT(DISTINCT user_id) AS active_users,
       AVG(success_flag::int) AS success_rate
FROM ml_raw_usage_events
GROUP BY 1 ORDER BY 1
""")
ax = monthly.set_index("month")["events"].plot(title="Raw ML events per month", marker=".")
ax.set_ylabel("events");

## 1. Tool adoption forecast — does a learnable trend exist?

Label: `label_adoption_rate` = distinct tool users / all employees per month.

In [ ]:
adoption = q("""
SELECT tool_name, forecast_month, label_adoption_rate
FROM ml_features_tool_adoption
ORDER BY tool_name, forecast_month
""")
pivot = adoption.pivot(index="forecast_month", columns="tool_name", values="label_adoption_rate")
ax = pivot.plot(title="Monthly adoption rate per tool (24 complete months)")
ax.set_ylabel("adoption rate");

In [ ]:
q("""
SELECT tool_name,
       MIN(label_adoption_rate) AS min_rate,
       MAX(label_adoption_rate) AS max_rate,
       ROUND((MAX(label_adoption_rate) - MIN(label_adoption_rate))::numeric, 4) AS lift,
       COUNT(*) AS months
FROM ml_features_tool_adoption
GROUP BY tool_name ORDER BY lift DESC
""")

## 2. User adoption risk — label balance & feature separation

Label: `label_risk_band` derived from month-over-month usage decline (`ml_raw_user_adoption_labels`).

In [ ]:
balance = q("""
SELECT label_risk_band, COUNT(*) AS rows
FROM ml_features_user_adoption_risk
WHERE label_risk_band IS NOT NULL
GROUP BY label_risk_band ORDER BY rows DESC
""")
balance.set_index("label_risk_band").plot.pie(y="rows", autopct="%1.1f%%", legend=False, title="Risk label balance")
plt.ylabel(""); balance

In [ ]:
q("""
SELECT label_risk_band,
       AVG(prev_month_events) AS avg_events,
       AVG(prev_month_distinct_tools) AS avg_tools,
       AVG(events_mom_delta) AS avg_mom_delta,
       AVG(prior6_mean_events) AS avg_prior6_events,
       AVG(label_usage_ratio) AS avg_next_month_ratio
FROM ml_features_user_adoption_risk
WHERE label_risk_band IS NOT NULL
GROUP BY label_risk_band
ORDER BY avg_events DESC
""")

## 3. User segmentation — clustering input shape

In [ ]:
seg = q("SELECT * FROM ml_features_user_segments")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
seg["total_events"].hist(bins=30, ax=axes[0]); axes[0].set_title("Events per user")
seg.plot.scatter("total_events", "distinct_tools", ax=axes[1], alpha=0.6,
                 title="Volume vs tool breadth")
seg[["total_events", "distinct_tools", "top_tool_share", "success_rate", "recency_days"]].describe()

## 4. Usage anomalies — is the spike signal visible?

`label_is_anomaly` marks weeks where surge events were injected (ground truth).

In [ ]:
anom = q("""
SELECT week_start, team_id, tool_id, actual_value, expected_value, z_score, label_is_anomaly
FROM ml_features_usage_anomalies
WHERE z_score IS NOT NULL
""")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
anom.plot.scatter("expected_value", "actual_value", s=8, ax=axes[0],
                  title="Actual vs 4-week expected")
anom[anom.label_is_anomaly].plot.scatter("expected_value", "actual_value", s=18,
                  color="red", ax=axes[0], label="injected")
axes[0].legend()
anom["z_score"].clip(-5, 8).hist(bins=60, ax=axes[1]); axes[1].set_title("z-score distribution")
print("anomaly prevalence:", round(anom.label_is_anomaly.mean(), 4))

## 5. Team productivity forecast — throughput history

In [ ]:
prod = q("""
SELECT team_name, metric_month, label_completed_story_points, prev_1_cycle
FROM ml_features_team_productivity
ORDER BY team_name, metric_month
""")
prod.pivot(index="metric_month", columns="team_name", values="label_completed_story_points").plot(
    title="Completed story points per team", legend=False, alpha=0.7)

In [ ]:
q("""
SELECT ROUND(CORR(label_completed_story_points, prev_1_points)::numeric, 3) AS corr_lag1,
       ROUND(CORR(label_completed_story_points, prev_1_cycle)::numeric, 3) AS corr_cycle_time,
       ROUND(CORR(label_completed_story_points, prev_1_story_count)::numeric, 3) AS corr_story_count
FROM ml_features_team_productivity
WHERE prev_1_points IS NOT NULL
""")

## 6. Tool recommendations — interaction density

In [ ]:
q("""
SELECT t.tool_name, t.category,
       SUM(r.usage_events) AS events,
       COUNT(DISTINCT r.user_id) AS users,
       AVG(r.success_rate) AS success_rate
FROM ml_features_tool_recommendations r
JOIN ai_tools t ON t.tool_id = r.tool_id
GROUP BY t.tool_id, t.tool_name, t.category
ORDER BY events DESC
""")

In [ ]:
q("""
SELECT COUNT(*) AS observed_pairs,
       COUNT(DISTINCT user_id) AS users,
       COUNT(DISTINCT tool_id) AS tools,
       ROUND(COUNT(*)::numeric / (COUNT(DISTINCT user_id) * COUNT(DISTINCT tool_id)), 3) AS density
FROM ml_features_tool_recommendations
""")

## Findings → modelling implications

1. **Adoption rates drift upward tool-by-tool** with stable seasonality → lag features + gradient boosting should forecast well (watch MAE ≈ 0.02–0.03 in the walkthrough).
2. **Risk labels are class-imbalanced and noisy by construction** (next-month volume is stochastic) → report macro-F1 vs the majority baseline, not accuracy alone.
3. **Behaviour features separate volume-driven segments** (long-tail user distribution) → k-means with silhouette-selected k works; names are assigned by interpretable rules.
4. **Injected surge weeks sit far above the 4-week baseline** (z up to ~7) → IsolationForest over the residual space beats a naive z ≥ 3 flag on precision.
5. **Throughput has trend + team idiosyncrasy** → recursive multi-step boosting with residual intervals.
6. **Interaction matrix is dense over long horizons** → recommendations use a 90-day window and engagement-based candidates, evaluated on *new-tool* adoption with precision@5.

Next: `01_training_walkthrough.ipynb` trains and evaluates every activity.